# 00 — Analyse exploratoire (EDA) avec graphes et interprétations

Objectif : **comprendre les données avant de modéliser**, vérifier visuellement chaque finding
de `FINDINGS.md`, et après chaque graphe expliquer ce qu'on en déduit pour la stratégie.

Plan :
1. Aperçu & intégrité (manquants, doublons, types)
2. La cible : où est la fraude ? (par opération)
3. La dimension temporelle (dérive, frontière train/test)
4. À l'intérieur d'op_03 : fraude vs légitime se ressemblent-elles ?
5. Le piège des soldes (arithmétique bruitée)
6. Le piège des comptes (mixtes) + graphe biparti
7. Pouvoir discriminant univarié (corrélations)
8. Synthèse → quelles features construire

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr

from src import config as C

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 4)
DATA = ROOT / "data"

train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")
print("train:", train.shape, "| test:", test.shape)

## 1. Aperçu & intégrité

In [ ]:
print("=== Types & aperçu ===")
display(train.dtypes.to_frame("dtype"))
print("Valeurs manquantes (train):", int(train.isna().sum().sum()))
print("Valeurs manquantes (test) :", int(test.isna().sum().sum()))
print("Doublons (train, hors id) :", int(train.drop(columns=[C.ID]).duplicated().sum()))
print("Montants <= 0             :", int((train[C.AMOUNT] <= 0).sum()))
print("Soldes origine négatifs   : %.2f%%" % (100 * (train[C.ORIGIN_BAL_BEFORE] < 0).mean()))
display(train[[C.AMOUNT, C.ORIGIN_BAL_BEFORE, C.ORIGIN_BAL_AFTER, C.DEST_BAL_BEFORE, C.DEST_BAL_AFTER]].describe())

**Interprétation.** On vérifie que les données sont saines (0 manquant, pas de doublon).
Si des soldes sont négatifs (~2,5% attendus), ce n'est **pas un bug** : c'est un artefact
d'anonymisation/rééchelonnement. Ne pas "corriger" ces valeurs.

## 2. La cible : où est la fraude ?

In [ ]:
by_op = train.groupby(C.OPERATION)[C.TARGET].agg(taux="mean", nb_fraudes="sum", nb_tx="count")
display(by_op)

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
by_op["taux"].plot.bar(ax=ax[0], color="#b3242b")
ax[0].set_title("Taux de fraude par opération")
ax[0].set_ylabel("taux de fraude")
(by_op["nb_tx"] / 1e3).plot.bar(ax=ax[1], color="#4c6ef5")
ax[1].set_title("Volume de transactions (milliers)")
ax[1].set_ylabel("nb tx (k)")
plt.tight_layout(); plt.show()

share = by_op["nb_fraudes"] / by_op["nb_fraudes"].sum()
print("Part des fraudes par opération:\n", (100 * share).round(1).astype(str) + " %")

**Interprétation.** Si 100% de la fraude est dans `op_03` (taux interne ~31%, les autres à 0%),
alors **prédire 0 hors op_03 est gratuit et correct**, et tout l'effort se concentre sur ~415k op_03.
Conséquence métrique : un classement "op_03 = risqué" donne déjà **AP ≈ 0,31**. Le reste se gagne
dans le classement *interne* à op_03 — c'est là que tout se joue.

## 3. La dimension temporelle

In [ ]:
rate = train.groupby(C.PERIOD)[C.TARGET].mean()
vol = train.groupby(C.PERIOD).size()

fig, ax1 = plt.subplots(figsize=(11, 4))
ax1.plot(rate.index, rate.values, color="#b3242b", label="taux de fraude")
ax1.axvspan(test[C.PERIOD].min(), test[C.PERIOD].max(), color="orange", alpha=0.15)
ax1.set_xlabel("period"); ax1.set_ylabel("taux de fraude", color="#b3242b")
ax1.set_title("Taux de fraude au fil du temps (zone orange = plage du test)")
ax1.axvline(train[C.PERIOD].max(), ls="--", color="gray")
plt.tight_layout(); plt.show()

print("train period:", train[C.PERIOD].min(), "->", train[C.PERIOD].max())
print("test  period:", test[C.PERIOD].min(), "->", test[C.PERIOD].max())
print("taux de fraude: min %.3f / max %.3f" % (rate.min(), rate.max()))

**Interprétation.** Le test est **strictement après** le train (aucun chevauchement) et le taux
de fraude varie fortement dans le temps. Deux conséquences :
- **Validation temporelle obligatoire** : une CV aléatoire mélange passé/futur et gonfle le score.
- **Dérive** : un modèle entraîné sur le passé se dégrade sur les périodes récentes. Le fold le plus
  récent est le meilleur proxy du score réel sur le test (cf. notebook 01, l'AP baisse fold après fold).

## 4. À l'intérieur d'op_03 : fraude vs légitime
On se restreint à op_03 (le seul terrain qui compte) et on compare les deux classes.

In [ ]:
op03 = train[train[C.OPERATION] == C.FRAUD_OPERATION].copy()
op03["amount_log1p"] = np.log1p(op03[C.AMOUNT].clip(lower=0))

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for flag, color, lbl in [(0, "#4c6ef5", "légitime"), (1, "#b3242b", "fraude")]:
    sub = op03[op03[C.TARGET] == flag]
    sns.kdeplot(sub["amount_log1p"], ax=ax[0], label=lbl, color=color, fill=True, alpha=0.3)
    sns.kdeplot(sub[C.ORIGIN_BAL_BEFORE].clip(-1e4, 1e6), ax=ax[1], label=lbl, color=color, fill=True, alpha=0.3)
ax[0].set_title("Montant (log) — op_03"); ax[0].legend()
ax[1].set_title("Solde émetteur avant — op_03"); ax[1].legend()
plt.tight_layout(); plt.show()

# montant rapporté à l'habitude du compte
acct_mean = op03.groupby(C.ORIGIN_ACCT)[C.AMOUNT].transform("mean")
op03["amount_vs_acct_mean"] = op03[C.AMOUNT] / (acct_mean + 1e-6)
print("Médiane montant/moyenne-compte — fraude : %.3f" % op03.loc[op03[C.TARGET]==1, "amount_vs_acct_mean"].median())
print("Médiane montant/moyenne-compte — légit  : %.3f" % op03.loc[op03[C.TARGET]==0, "amount_vs_acct_mean"].median())

**Interprétation.** Si les deux courbes se superposent largement, c'est la confirmation visuelle
que **aucune variable brute ne sépare** fraude et légitime dans op_03. Le montant des fraudes n'est
pas anormal vs l'habitude du compte (médianes quasi identiques). → Le signal est **multivarié** et
**comportemental** : il faudra des agrégations (par compte, par couple, dans le temps), pas des colonnes brutes.

## 5. Le piège des soldes (arithmétique bruitée)

In [ ]:
# égalité attendue : solde_après = solde_avant - montant (côté émetteur)
expected_after = op03[C.ORIGIN_BAL_BEFORE] - op03[C.AMOUNT]
inconsistency = (op03[C.ORIGIN_BAL_AFTER] - expected_after).abs()
op03["balance_inconsistent"] = (inconsistency > 1.0).astype(int)

print("%% de transactions à arithmétique incohérente : %.1f%%" % (100 * op03["balance_inconsistent"].mean()))
rho, _ = spearmanr(op03["balance_inconsistent"], op03[C.TARGET])
print("Corrélation (Spearman) incohérence <-> fraude : %.3f" % rho)

ct = pd.crosstab(op03["balance_inconsistent"], op03[C.TARGET], normalize="index")
display(ct.rename(columns={0: "légit", 1: "fraude"}))

**Interprétation — DÉCOUVERTE qui contredit le rapport global.** Dans op_03, l'incohérence
d'arithmétique de solde n'est PAS un piège : seulement ~4% des tx sont incohérentes, mais elles
sont frauduleuses à **~56%** (contre ~30% en moyenne), et c'est le signal univarié le **plus fort**
(Spearman ≈ 0,11). Le rapport mesurait sur tout le dataset (≈40% incohérent, corrélation diluée à
≈0,04 par les opérations sans fraude). Leçon : toujours revérifier dans le bon périmètre. → On **garde**
`balance_inconsistent` (+ variantes magnitude/signe, côtés émetteur et destinataire) comme feature,
implémentée dans `src/features/temporal.py:balance_features` et mesurée dans le notebook 01.

## 6. Le piège des comptes (mixtes) + graphe biparti

In [ ]:
# 6a. Graphe biparti ? un compte est-il à la fois émetteur ET destinataire ?
origins = set(train[C.ORIGIN_ACCT].unique())
dests = set(train[C.DEST_ACCT].unique())
print("Comptes émetteurs:", len(origins), "| destinataires:", len(dests))
print("Comptes jouant les DEUX rôles:", len(origins & dests), "-> graphe biparti si 0")

# 6b. Les comptes émetteurs sont-ils mixtes (fraude ET légitime) ?
g = op03.groupby(C.ORIGIN_ACCT)[C.TARGET].agg(["mean", "count"])
profil = pd.cut(g["mean"], [-0.01, 0.0, 0.999, 1.01],
                labels=["jamais fraude", "mixte", "toujours fraude"])
counts = profil.value_counts()
tx_share = op03.groupby(C.ORIGIN_ACCT).size().groupby(profil).sum()

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
counts.plot.bar(ax=ax[0], color=["#2f9e44", "#4c6ef5", "#b3242b"])
ax[0].set_title("Nb de comptes émetteurs par profil")
(100 * tx_share / tx_share.sum()).plot.bar(ax=ax[1], color=["#2f9e44", "#4c6ef5", "#b3242b"])
ax[1].set_title("% des transactions op_03 par profil de compte")
plt.tight_layout(); plt.show()
print("%% tx venant de comptes MIXTES : %.1f%%" % (100 * tx_share.get("mixte", 0) / tx_share.sum()))

**Interprétation.** Deux conclusions majeures :
- **Graphe biparti** (0 compte dans les deux rôles) → pas de cycles, pas de mules. PageRank/Louvain/GNN
  ne sont **pas justifiés** ; seules des features de **degré bipartite** (collecteur / fan-out) ont du sens.
- **Comptes mixtes** (la grande majorité des tx viennent de comptes faisant fraude *et* légitime) →
  encoder l'**ID de compte** (target encoding) donne une CV flatteuse mais **ne généralise pas**.
  C'est précisément le surapprentissage des identifiants à éviter.

## 7. Pouvoir discriminant univarié (corrélations)

In [ ]:
op03["amount_vs_origin_before"] = op03[C.AMOUNT] / (op03[C.ORIGIN_BAL_BEFORE].abs() + 1e-6)
op03["dest_tx_count"] = op03.groupby(C.DEST_ACCT)[C.AMOUNT].transform("count")
cols = [C.AMOUNT, "amount_log1p", C.ORIGIN_BAL_BEFORE, "amount_vs_origin_before",
        "amount_vs_acct_mean", "balance_inconsistent", "dest_tx_count"]
corr = {c: spearmanr(op03[c], op03[C.TARGET])[0] for c in cols}
corr = pd.Series(corr).sort_values(key=np.abs, ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
corr.plot.barh(ax=ax, color=np.where(corr > 0, "#b3242b", "#4c6ef5"))
ax.set_title("Corrélation Spearman avec la fraude (op_03)"); ax.invert_yaxis()
plt.tight_layout(); plt.show()
display(corr.to_frame("spearman").round(3))

**Interprétation.** Si toutes les |corrélations| restent sous ~0,1, aucune variable seule ne suffit :
le modèle devra **combiner** des signaux faibles (interactions) — exactement ce que font CatBoost/LightGBM.
Ça confirme que l'avantage viendra du **feature engineering comportemental**, pas du choix du modèle.

## 8. Synthèse — quelles features construire

| Constat EDA | Action features |
|---|---|
| Fraude 100% dans op_03 | Restreindre la modélisation à op_03 (déjà fait notebook 01) |
| Classes se ressemblent en univarié | Features **multivariées / comportementales** |
| Dérive temporelle | Features de **dynamique récente** (fenêtres glissantes), validation temporelle |
| Comptes mixtes | **Pas** d'encodage d'ID brut ; agrégations comportementales fold-safe |
| **Incohérence de solde (DANS op_03)** | **Signal le plus fort en univarié → à garder** (`balance_features`) |
| Graphe biparti | Degrés bipartites légers (collecteur / fan-out), pas de GNN |

**Pistes prioritaires** : incohérence de solde (émetteur + destinataire), fréquence/montant typique du
couple (émetteur, destinataire), nouveauté de la relation, intervalle depuis la dernière tx, comptage
récent par compte, degré entrant du destinataire.